# 编码器生成上下文向量
# 解码器则是预测下一个词

In [8]:
import torch
import torch.nn as nn
import random
torch.manual_seed(42)  # 设置随机种子以确保结果可复现
batch_size = 8
src_len = 10
trg_len = 12
src_vocab_size = 100
trg_vocab_size = 120
hidden_size = 64
num_layers = 2
sos_idx = 1
eos_idx = 2

class Encoder(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_layers):
        super(Encoder,self).__init__()
        self.embedding = nn.Embedding(vocab_size,hidden_size)# 输入词嵌入
        self.rnn = nn.LSTM(
            hidden_size = hidden_size,
            input_size = hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False
        )
    def forward(self,x):#x为（batch_size,seq_len)
        x = self.embedding(x)#输出后x为(batch_size,seq_len,hidden_size)
        _,(h_n,c_n) = self.rnn(x)#outputs包含了每一个隐藏状态
        return h_n,c_n
    
class Decoder(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_layers):
        super(Decoder,self).__init__()
        self.embedding = nn.Embedding(vocab_size,hidden_size)
        self.rnn = nn.LSTM(
            hidden_size = hidden_size,
            input_size = hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size,vocab_size)
    def forward(self,x,hidden,cell):#这个x为(batch_size)
        x.unsqueeze_(1)  # 加上长度维度
        x = self.embedding(x)
        output,(h_n,c_n) = self.rnn(x,(hidden,cell))
        output.squeeze_(1)  # 去掉长度维度
        output= self.fc(output)
        return output,h_n,c_n
    
#采用教师强制    
class Seq2Seq(nn.Module):
    def __init__(self,encoder,decoder,device):
        super(Seq2Seq,self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self,src,trg,teacher_forcing_ratio = 0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size,trg_len,trg_vocab_size).to(self.device)
        hidden,cell = self.encoder(src)
        input = trg[:,0]  # 第一个输入是<sos>标记
        for t in range(1,trg_len):
            output,hidden,cell = self.decoder(input,hidden,cell)
            outputs[:,t,:] = output
            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(1)
            input = trg[:,t] if teacher_force else top1
        return outputs
    
    def greedy_decode(self,src,max_len=12,sos_idx=1,eos_idx=2):
        self.eval()
        with torch.no_grad():
            hidden,cell = self.encoder(src)
            trg_indexes = [sos_idx]
            for _ in range(1,max_len):
                input = torch.LongTensor([trg_indexes[-1]]).to(self.device)
                output,hidden,cell = self.decoder(input,hidden,cell)
                pred_token = output.argmax(1).item()
                trg_indexes.append(pred_token)
                if pred_token == eos_idx:
                    break
        return trg_indexes

class DecoderAlt(nn.Module):
    def __init__(self,vacab_size,hidden_size,num_layers):
        super(DecoderAlt,self).__init__()
        self.embedding = nn.Embedding(vacab_size,hidden_size)
        self.rnn = nn.LSTM(
            hidden_size = hidden_size,
            input_size = hidden_size+hidden_size,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size,vacab_size)
    def forward(self,x,hidden_ctx,hidden,cell):#这个x为(batch_size)
        x=x.unsqueeze(1)  # 加上长度维度
        embedded = self.embedding(x)
        context = hidden_ctx[-1].unsqueeze(1).repeat(1,embedded.shape[1],1)# 复制上下文向量以匹配嵌入的时间步长
        rnn_input = torch.cat((embedded,context),dim=2)
        outputs,(h_n,c_n) = self.rnn(rnn_input,(hidden,cell))
        outputs.squeeze_(1)  # 去掉长度维度
        outputs= self.fc(outputs)
        return outputs,h_n,c_n
    
def alternative_greedy_decode(encoder,decoder,src,device,max_len=trg_len):
    with torch.no_grad():
        hidden_ctx,cell_ctx = encoder(src)
        hidden,cell = hidden_ctx,cell_ctx
        trg_indexes = [sos_idx]
        for _ in range(1,max_len):
            input = torch.LongTensor([trg_indexes[-1]]).to(device)
            output,hidden,cell = decoder(input,hidden_ctx,hidden,cell)
            pred_token = output.argmax(1).item()
            trg_indexes.append(pred_token)
            if pred_token == eos_idx:
                break      
    return trg_indexes  
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available()else "cpu")
    encoder = Encoder(src_vocab_size,hidden_size,num_layers).to(device)
    decoder = Decoder(trg_vocab_size,hidden_size,num_layers).to(device)
    model = Seq2Seq(encoder,decoder,device).to(device)
    src = torch.randint(1,src_vocab_size,(batch_size,src_len)).to(device)
    trg = torch.randint(1,trg_vocab_size,(batch_size,trg_len)).to(device)
    model.train()
    print(model(src,trg).shape)
    decoder_alt = DecoderAlt(trg_vocab_size,hidden_size,num_layers).to(device)
    print(alternative_greedy_decode(encoder,decoder_alt,src[0:1,:],device))

torch.Size([8, 12, 120])
[1, 75, 13, 13, 45, 13, 45, 13, 13, 45, 13, 45]


# 注意力机制
$$
C_t = \sum_{j=1}^{T_x} \alpha_{tj} h_j
$$

## 注意力计算
### 计算相似度
#### 解码器上一隐藏状态与编码器所有隐藏状态计算相似度
$$
e_{tj} = \text{score}(h^{\prime}_{t-1}, h_j)
$$

### 计算权重
$$
\alpha_{tj} = \text{softmax}(e_{tj}) = \frac{\exp(e_{tj})}{\sum_{i=1}^{T_x} \exp(e_{ti})}
$$
### 加权求和
$$
C_t = \sum_{j=1}^{T_x} \alpha_{tj} h_j
$$

### QKV 机制
$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

In [ ]:
import torch
import numpy as np
import torch.nn as nn
import random
torch.manual_seed(42)  # 设置随机种子以确保结果可复现
batch_size = 8
src_len = 10
trg_len = 12
src_vocab_size = 100
trg_vocab_size = 120
hidden_size = 64
num_layers = 2
sos_idx = 1
eos_idx = 2

class Encoder(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_layers):
        super(Encoder,self).__init__()
        self.embedding = nn.Embedding(vocab_size,hidden_size)
        self.rnn = nn.LSTM(
            hidden_size = hidden_size,
            input_size = hidden_size,
            num_layers = num_layers,
            batch_first = True,
            bidirectional = True
        )
        self.fc = nn.Linear(hidden_size*2,hidden_size)
    def forward(self,x):
        x = self.embedding(x)
        outputs,(h_n,c_n) = self.rnn(x)
        outputs = self.fc(outputs)
        outputs = torch.tanh(outputs)
        return outputs,h_n,c_n#(batch_size,src_len,hidden_size)
    
class AttentionSimple(nn.Module):#直接相乘
    def __init__(self,hidden_size):
        super(AttentionSimple,self).__init__()
        self.register_buffer("scale_factor",torch.sqrt(torch.FloatTensor([hidden_size])))
    def forward(self,hidden,encoder_outputs):
        query = hidden[-1].unsqueeze(1)#(batch_size,1,hidden_size)
        keys = encoder_outputs #(batch_size,src_len,hidden_size)
        energy = torch.bmm(query,keys.transpose(1,2))#(batch_size,1,src_len)
        return torch.softmax(energy/self.scale_factor,dim=2)  #(batch_size,1,src_len)对src每个位置的关注程度
        
class AttentionParams(nn.Module):
    def __init__(self,hidden_size):
        super(AttentionParams,self).__init__()
        self.attn = nn.Linear(hidden_size*2,hidden_size)#通过小型神经网路拉学习Query和Key的关系
        self.v = nn.Parameter(torch.rand(hidden_size))
    def forward(self,hidden,encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden = hidden[-1].unsqueeze(1).repeat(1,src_len,1)
        energy = self.attn(torch.cat((hidden,encoder_outputs),dim=2))
        attention = torch.sum(self.v * torch.tanh(energy),dim=2)
        return torch.softmax(attention,dim=1).unsqueeze(1)#(batch_size,1,src_len)
    
class DecoderWithAttention(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_layers,attention_module):
        super(DecoderWithAttention,self).__init__()
        self.attention = attention_module
        self.embedding = nn.Embedding(vocab_size,hidden_size)
        self.rnn=nn.LSTM(
            hidden_size = hidden_size,
            input_size = hidden_size + hidden_size,
            num_layers = num_layers,
            batch_first = True,
        )
        self.fc = nn.Linear(hidden_size,vocab_size)
    def forward(self,x,hidden,cell,encoder_outputs):
        embedded = self.embedding(x.unsqueeze(1))
        a=self.attention(hidden,encoder_outputs)
        context = torch.bmm(a,encoder_outputs)
        rnn_input = torch.cat((embedded,context),dim=2)
        output,(h_n,c_n) = self.rnn(rnn_input,(hidden,cell))
        output = self.fc(output.squeeze(1))
        return output,h_n,c_n#h_n:(num_layers,batch_size,hidden_size)
        
class Seq2SeqWithAttention(nn.Module):
    def __init__(self,encoder,decoder,device):
        super(Seq2SeqWithAttention,self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
    def forward(self,src,trg,teacher_forcing_ratio = 0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size,trg_len,trg_vocab_size).to(self.device)
        encoder_outputs,hidden,cell = self.encoder(src)
        hidden = hidden.view(self.encoder.rnn.num_layers,2,batch_size,-1).sum(1)
        cell = cell.view(self.encoder.rnn.num_layers,2,batch_size,-1).sum(1)
        input = trg[:,0]
        for t in range(1,trg_len):
            output,hidden,cell = self.decoder(input,hidden,cell,encoder_outputs)
            outputs[:,t,:] = output
            teacher_force = random.random() < teacher_forcing_ratio
            input = trg[:,t] if teacher_force else output.argmax(1)
        return outputs
    
if __name__ == '__main__':
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # --- 统一创建编码器和伪数据 ---
    encoder = Encoder(src_vocab_size, hidden_size, num_layers).to(device)
    src = torch.randint(1, src_vocab_size, (batch_size, src_len)).to(device)#（batch_size,src_len)
    trg = torch.randint(1, trg_vocab_size, (batch_size, trg_len)).to(device)

    # =========================================
    # 1: 无参数的基础 Attention
    # =========================================
    print("\n" + "="*20 + " 1: 无参数的基础 Attention " + "="*20)
    attention_simple = AttentionSimple(hidden_size).to(device)
    decoder_simple = DecoderWithAttention(trg_vocab_size, hidden_size, num_layers, attention_simple).to(device)
    model_simple = Seq2SeqWithAttention(encoder, decoder_simple, device).to(device)
    
    model_simple.train()
    outputs_simple = model_simple(src, trg)
    print(f"模型结构:\n{model_simple}")
    print(f"\n训练模式输出张量形状: {outputs_simple.shape}")
    


    # =========================================
    # 2: 带参数的 Attention
    # =========================================
    print("\n" + "="*20 + " 2: 带参数的 Attention " + "="*20)
    attention_params = AttentionParams(hidden_size).to(device)
    decoder_params = DecoderWithAttention(trg_vocab_size, hidden_size, num_layers, attention_params).to(device)
    model_params = Seq2SeqWithAttention(encoder, decoder_params, device).to(device)

    model_params.train()
    outputs_params = model_params(src, trg)
    print(f"模型结构:\n{model_params}")
    print(f"\n训练模式输出张量形状: {outputs_params.shape}")
    
        

Using device: cuda

==================== 1: 无参数的基础 Attention ====================
模型结构:
Seq2SeqWithAttention(
  (encoder): Encoder(
    (embedding): Embedding(100, 64)
    (rnn): LSTM(64, 64, num_layers=2, batch_first=True, bidirectional=True)
    (fc): Linear(in_features=128, out_features=64, bias=True)
  )
  (decoder): DecoderWithAttention(
    (attention): AttentionSimple()
    (embedding): Embedding(120, 64)
    (rnn): LSTM(128, 64, num_layers=2, batch_first=True)
    (fc): Linear(in_features=64, out_features=120, bias=True)
  )
)

训练模式输出张量形状: torch.Size([8, 12, 120])

==================== 2: 带参数的 Attention ====================
模型结构:
Seq2SeqWithAttention(
  (encoder): Encoder(
    (embedding): Embedding(100, 64)
    (rnn): LSTM(64, 64, num_layers=2, batch_first=True, bidirectional=True)
    (fc): Linear(in_features=128, out_features=64, bias=True)
  )
  (decoder): DecoderWithAttention(
    (attention): AttentionParams(
      (attn): Linear(in_features=128, out_features=64, bias=Tru

# Transformer
## 自注意力
$$
q_i = x_i W^Q \\
k_i = x_i W^K \\
v_i = x_i W^V
$$


In [18]:
import torch
import torch.nn as nn
import math
class SelfAttention(nn.Module):
    def __init__(self,hidden_size):
         super(SelfAttention,self).__init__()
         self.hidden_size = hidden_size
         self.k_linear = nn.Linear(hidden_size,hidden_size)
         self.q_linear = nn.Linear(hidden_size,hidden_size)
         self.v_linear = nn.Linear(hidden_size,hidden_size)
    def forward(self,x):
        q = self.q_linear(x)
        k = self.k_linear(x)
        v = self.v_linear(x)
        scores = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.hidden_size)#(batch_size,seq_len,seq_len)
        attention = torch.softmax(scores,dim=-1)
        context = torch.matmul(attention,v)#(batch_size,seq_len,hidden_size)
        return context
    
class MultiHeadAttention(nn.Module):
    def __init__(self,hidden_size,num_heads):
        super(MultiHeadAttention,self).__init__()
        assert hidden_size % num_heads ==0,"hidden_size must be divisible by num_heads"
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_size = hidden_size // num_heads#每个头的维度
        
        self.q_linear = nn.Linear(hidden_size,hidden_size)
        self.k_linear = nn.Linear(hidden_size,hidden_size)
        self.v_linear = nn.Linear(hidden_size,hidden_size)
        self.fc = nn.Linear(hidden_size,hidden_size)
    def forward(self,x):
        batch_size,seq_len,_ = x.size()
        q = self.q_linear(x)
        k = self.k_linear(x)
        v = self.v_linear(x)
        q = q.view(batch_size,seq_len,self.num_heads,self.head_size).transpose(1,2)#(batch_size,num_heads,seq_len,head_size)
        k = k.view(batch_size,seq_len,self.num_heads,self.head_size).transpose(1,2)
        v = v.view(batch_size,seq_len,self.num_heads,self.head_size).transpose(1,2)
        scores = torch.matmul(q,k.transpose(-2,-1))/math.sqrt(self.head_size)#(batch_size,num_heads,seq_len,seq_len)
        attention = torch.softmax(scores,dim=-1)
        context = torch.matmul(attention,v)#(batch_size,num_heads,seq_len,head_size)
        context = context.transpose(1,2).contiguous().view(batch_size,seq_len,self.hidden_size)#.contiguous()保持内存连续
        output = self.fc(context)
        return output
if __name__ == '__main__':
    # --- 全局参数 ---
    batch_size = 8
    seq_len = 10
    hidden_size = 512
    num_heads = 8

    # --- 伪数据 ---
    # 创建一个随机输入张量
    input_tensor = torch.rand(batch_size, seq_len, hidden_size)
    print(f"输入张量形状: {input_tensor.shape}\n")

    # =========================================
    # 1: 单头自注意力
    # =========================================
    print("="*20 + " 1: 测试单头自注意力 " + "="*20)
    self_attention = SelfAttention(hidden_size)
    output_self = self_attention(input_tensor)
    print(f"单头自注意力输出形状: {output_self.shape}\n")

    # =========================================
    # 2: 多头自注意力
    # =========================================
    print("="*20 + " 2: 测试多头自注意力 " + "="*20)
    multi_head_attention = MultiHeadAttention(hidden_size, num_heads)
    output_multi_head = multi_head_attention(input_tensor)
    print(f"多头自注意力输出形状: {output_multi_head.shape}")                 

输入张量形状: torch.Size([8, 10, 512])

==================== 1: 测试单头自注意力 ====================
单头自注意力输出形状: torch.Size([8, 10, 512])

==================== 2: 测试多头自注意力 ====================
多头自注意力输出形状: torch.Size([8, 10, 512])
